In [ ]:
import os
import sys
import copy
import random
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Set styling and random seed for reproducibility
sns.set_theme(style="whitegrid")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")


In [ ]:
# Load the Cat & Dog dataset as specified in proj1.pdf
DATA_DIR = './catdog_data'
if 'google.colab' in sys.modules:
    DATA_DIR = '/content/catdog_data'
    if not os.path.exists(DATA_DIR):
        !wget -q https://github.com/johannm00-ux/Deep-Learning-Project-1/releases/download/v1.0/catdog_data.zip -O /content/catdog_data.zip
        with zipfile.ZipFile('/content/catdog_data.zip', 'r') as zip_ref:
            zip_ref.extractall('/content/')

cats_dir = os.path.join(DATA_DIR, 'cats')
dogs_dir = os.path.join(DATA_DIR, 'dogs')

cat_files = sorted([os.path.join(cats_dir, f) for f in os.listdir(cats_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
dog_files = sorted([os.path.join(dogs_dir, f) for f in os.listdir(dogs_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

classes = ["cat", "dog"]
all_files = cat_files + dog_files
all_labels = [0] * len(cat_files) + [1] * len(dog_files)


## 1. Task 1: Split the Data and Create the DataLoader


In [ ]:
# 1.a: Inspect raw image dimensions and split into Train (70%), Validation (15%), and Test (15%)
raw_sizes = [Image.open(f).size for f in all_files]
widths, heights = zip(*raw_sizes)

trainval_files, test_files, trainval_labels, test_labels = train_test_split(
    all_files, all_labels, test_size=0.15, stratify=all_labels, random_state=SEED
)
train_files, val_files, train_labels, val_labels = train_test_split(
    trainval_files, trainval_labels, test_size=0.15 / 0.85, stratify=trainval_labels, random_state=SEED
)

print(f"Total dataset images      : {len(all_files):,} ({len(cat_files):,} cats, {len(dog_files):,} dogs)")
print(f"Unique raw resolutions    : {len(set(raw_sizes)):,}")
print(f"Raw width range (min/max) : {min(widths)} to {max(widths)} px (mean: {np.mean(widths):.1f} px)")
print(f"Raw height range (min/max): {min(heights)} to {max(heights)} px (mean: {np.mean(heights):.1f} px)")
print(f"Train split (70%)         : {len(train_files):,} images")
print(f"Validation split (15%)    : {len(val_files):,} images")
print(f"Test split (15%)          : {len(test_files):,} images")


### 1.b Class Distribution Across Splits


In [ ]:
train_counts = np.bincount(train_labels, minlength=2)
val_counts = np.bincount(val_labels, minlength=2)
test_counts = np.bincount(test_labels, minlength=2)

fig, ax = plt.subplots(figsize=(8, 5))
bar_width = 0.25
x = np.arange(len(classes))

ax.bar(x - bar_width, train_counts, width=bar_width, label='Train (70%)', color='#3470a3', edgecolor='black', alpha=0.85)
ax.bar(x, val_counts, width=bar_width, label='Validation (15%)', color='#2ca02c', edgecolor='black', alpha=0.85)
ax.bar(x + bar_width, test_counts, width=bar_width, label='Test (15%)', color='#e76f51', edgecolor='black', alpha=0.85)

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Class Distribution Across Train, Validation, and Test Splits', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Cat (0)', 'Dog (1)'])
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


### 1.c Custom Dataset and On-the-Fly Data Augmentation


In [ ]:
# 1.c: Define Dataset and DataLoaders with on-the-fly training augmentation
image_size = 128
batch_size = 64

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

class CatDogDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

train_dataset = CatDogDataset(train_files, train_labels, transform=train_transform)
val_dataset = CatDogDataset(val_files, val_labels, transform=eval_transform)
test_dataset = CatDogDataset(test_files, test_labels, transform=eval_transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Sanity check batch shapes and plot sample augmented images
images, labels = next(iter(train_loader))
print(f"Feature batch shape : {images.shape}")
print(f"Labels batch shape  : {labels.shape}")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i].permute(1, 2, 0).numpy())
    ax.set_title(f"Label: {labels[i].item()} ({classes[labels[i].item()]})", fontsize=10)
    ax.axis('off')

plt.suptitle("Sample Augmented Training Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Task 2: Constructing and Training the Network

### 2.a Baseline Fully Connected Neural Network (FCNN)


In [ ]:
# 2.a: Baseline FCNN for comparison against CNN
class BaselineFCNN(nn.Module):
    def __init__(self, in_size: int = 3 * 128 * 128, out_size: int = 2):
        super(BaselineFCNN, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_1 = nn.Linear(in_features=in_size, out_features=512)
        self.linear_2 = nn.Linear(in_features=512, out_features=512)
        self.linear_3 = nn.Linear(in_features=512, out_features=out_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        x = F.relu(self.linear_1(x))
        x = F.relu(self.linear_2(x))
        x = self.linear_3(x)
        return x

fcnn_model = BaselineFCNN().to(device)
fcnn_params = sum(p.numel() for p in fcnn_model.parameters() if p.requires_grad)
print(fcnn_model)
print(f"\nBaseline FCNN trainable parameters: {fcnn_params:,}")


### 2.b Convolutional Neural Network (CNN) Architecture


In [ ]:
# 2.b: Define CNN architecture with BatchNorm, Dropout, and Global Average Pooling
input_dim = (128, 128)
channel_dim = 3

class CatDogCNN(nn.Module):
    def __init__(self, dropout_rate: float = 0.3):
        super(CatDogCNN, self).__init__()
        # Block 1: 3 x 128 x 128 -> 16 x 64 x 64
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 2: 16 x 64 x 64 -> 32 x 32 x 32
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 3: 32 x 32 x 32 -> 64 x 16 x 16
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 4: 64 x 16 x 16 -> 128 x 8 x 8
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Global Average Pooling & Classifier: 128 x 8 x 8 -> 128 x 1 x 1 -> 2
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc = nn.Linear(128, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = CatDogCNN().to(device)
cnn_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nCatDogCNN trainable parameters: {cnn_params:,}")

initial_conv1_weights = model.conv1.weight.clone().detach().cpu()


### 2.c Training the CNN


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 25
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print(f"Starting training on {device} for {num_epochs} epochs...")

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100.0

    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100.0

    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, "competition.pth")

    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] | "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

# Load best model weights
model.load_state_dict(best_model_wts)
print(f"\nBest Validation Accuracy: {best_val_acc:.2f}% (saved to competition.pth)")


## 3. Task 3: Visualizing Your Results

### 3.a Training vs. Validation Loss and Accuracy


In [ ]:
epochs_range = range(1, num_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#1f77b4', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 's--', label='Validation Loss', color='#d62728', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax1.set_title('Training vs. Validation Loss', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(epochs_range, history['train_acc'], 'o-', label='Train Accuracy', color='#1f77b4', linewidth=2)
ax2.plot(epochs_range, history['val_acc'], 's--', label='Validation Accuracy', color='#2ca02c', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training vs. Validation Accuracy', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


### 3.b First Convolutional Layer Filters: Before vs. After Training


In [ ]:
# 3.b: Plot first conv layer filters before and after training
trained_conv1_weights = model.conv1.weight.clone().detach().cpu()

num_filters = initial_conv1_weights.shape[0]

fig = plt.figure(figsize=(16, 9))
fig.suptitle('First Convolutional Layer (Conv1) Filters: Initial (Random) vs. Trained', fontsize=15, fontweight='bold', y=0.98)

# Normalize RGB filter weights to [0, 1] for visualization
def norm_filter(w):
    w_min, w_max = w.min(), w.max()
    return (w - w_min) / (w_max - w_min + 1e-8)

gs_before = fig.add_gridspec(2, 8, top=0.90, bottom=0.55)
gs_after  = fig.add_gridspec(2, 8, top=0.45, bottom=0.10)

fig.text(0.5, 0.93, '--- BEFORE TRAINING (Random Initialization) ---', ha='center', fontsize=12, fontweight='bold', color='#444444')
fig.text(0.5, 0.48, '--- AFTER TRAINING (Learned Filters) ---', ha='center', fontsize=12, fontweight='bold', color='#117733')

for i in range(num_filters):
    r = i // 8
    c = i % 8

    ax_b = fig.add_subplot(gs_before[r, c])
    ax_b.imshow(norm_filter(initial_conv1_weights[i]).permute(1, 2, 0).numpy())
    ax_b.set_title(f"Init #{i}", fontsize=9)
    ax_b.axis('off')

    ax_a = fig.add_subplot(gs_after[r, c])
    ax_a.imshow(norm_filter(trained_conv1_weights[i]).permute(1, 2, 0).numpy())
    ax_a.set_title(f"Trained #{i}", fontsize=9)
    ax_a.axis('off')

plt.show()


### 3.c Feeding an Image to the First Convolutional Layer and Plotting Activations


In [ ]:
# 3.c: Feed a sample test image to the first conv layer and plot its activations
model.eval()

sample_idx = 0
sample_image, sample_label = test_dataset[sample_idx]
sample_tensor = sample_image.unsqueeze(0).to(device)  # shape: (1, 3, 128, 128)

with torch.no_grad():
    conv1_acts = F.relu(model.bn1(model.conv1(sample_tensor))).squeeze(0).cpu()  # shape: (16, 128, 128)

fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(4, 6)

# Plot original input image spanning all 4 rows in cols 0 and 1
ax_orig = fig.add_subplot(gs[:, :2])
ax_orig.imshow(sample_image.permute(1, 2, 0).numpy())
ax_orig.set_title(f"Input Image (Label: {classes[sample_label]})\nShape: {tuple(sample_image.shape)}", fontsize=14, fontweight='bold')
ax_orig.axis('off')

# Plot all 16 feature maps across 4 rows and 4 columns (cols 2 through 5)
for i in range(16):
    r = i // 4
    c = 2 + (i % 4)
    ax_act = fig.add_subplot(gs[r, c])
    act_map = conv1_acts[i].numpy()
    ax_act.imshow(act_map, cmap='viridis')
    ax_act.set_title(f"Filter #{i} Activation", fontsize=9)
    ax_act.axis('off')

plt.suptitle(f"First Convolutional Layer Activations for '{classes[sample_label]}'", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


### 3.d Test Set Evaluation, Confusion Matrix, and Incorrectly Labelled Data Points


In [ ]:
# 3.d: Evaluate on test set and plot misclassified data points
model.eval()

all_preds = []
all_targets = []
misclassified_samples = []

with torch.no_grad():
    for images, labels in test_loader:
        images_dev = images.to(device)
        outputs = model(images_dev)
        probs = F.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

        for img, true_lbl, pred_lbl, prob in zip(images, labels, preds.cpu(), probs.cpu()):
            if pred_lbl != true_lbl:
                misclassified_samples.append({
                    'image': img.permute(1, 2, 0).numpy(),
                    'true_label': classes[true_lbl.item()],
                    'pred_label': classes[pred_lbl.item()],
                    'pred_conf': prob[pred_lbl].item() * 100.0,
                    'true_conf': prob[true_lbl].item() * 100.0
                })

total_test = len(test_dataset)
num_errors = len(misclassified_samples)
accuracy = 100.0 * (total_test - num_errors) / total_test

print(f"Total test samples       : {total_test}")
print(f"Total misclassifications : {num_errors}")
print(f"Test Accuracy            : {accuracy:.2f}%\n")
print(classification_report(all_targets, all_preds, target_names=classes))

# Plot Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Test Set Confusion Matrix (Accuracy: {accuracy:.2f}%)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot 10 misclassified examples in a 2x5 grid
num_display = min(10, num_errors)
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()

for idx in range(num_display):
    sample = misclassified_samples[idx]
    ax = axes[idx]
    ax.imshow(sample['image'])
    ax.set_title(
        f"True: {sample['true_label']} | Pred: {sample['pred_label']}\n"
        f"Pred Conf: {sample['pred_conf']:.1f}%\n"
        f"True Conf: {sample['true_conf']:.1f}%",
        fontsize=10,
        color='darkred'
    )
    ax.axis('off')

for idx in range(num_display, len(axes)):
    axes[idx].axis('off')

plt.suptitle(f"Sample Incorrectly Labelled Test Data Points ({num_errors} total errors)", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 4. Competition Model Export


In [ ]:
# Export competition.py as specified in proj1.pdf
competition_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F

input_dim = (128, 128)
channel_dim = 3

class competition(nn.Module):
    def __init__(self, dropout_rate: float = 0.3):
        super(competition, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc = nn.Linear(128, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x
'''

with open("competition.py", "w") as f:
    f.write(competition_code)

import competition as comp_mod
comp_model = comp_mod.competition()
comp_model.load_state_dict(torch.load("competition.pth", map_location="cpu"))
comp_model.eval()

dummy_x = torch.randn(4, comp_mod.channel_dim, *comp_mod.input_dim)
print(f"Competition model verified! Output shape: {comp_model(dummy_x).shape}")
